# Camada Gold
---

In [0]:
# Criação do SCHEMA da camada Gold
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## 1. Criação da dimensão de Filmes
---

In [0]:
# Carregar a tabela da camada Silver
df_dim_movies = spark.table("workspace.silver.tb_info_filmes")

# Selecionar apenas as colunas que farão parte da dimensão.
df_dim_movies = df_dim_movies.select(
    "id_filme",
    "titulo",
    "data_lancamento",
    "ano_lancamento",
    "duracao_minutos",
    "idioma_original",
    "status_filme",
    "sinopse"
)

# Criação de uma chave substituta (Surrogate Key) para cada filme.
# Utilizarei monotonically_increasing_id.
janela_movies = Window.orderBy("id_filme")

df_dim_movies = df_dim_movies.withColumn(
    "sk_movie_id",
    F.monotonically_increasing_id()
)

# Reorganizamos as colunas para colocar a chave substituta primeiro.
df_dim_movies = df_dim_movies.select(
    "sk_movie_id",
    "id_filme",
    "titulo",
    "data_lancamento",
    "ano_lancamento",
    "duracao_minutos",
    "idioma_original",
    "status_filme",
    "sinopse"
)

display(df_dim_movies)
# Verificação dos tipos de cada coluna
print(df_dim_movies.dtypes)

df_dim_movies.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_movies")

## 2. Criação da dimensão de Gêneros
---

In [0]:
# Carregar a tabela da camada Silver
df_dim_genres = (
    spark.table("workspace.silver.tb_generos")
    .select("nome_genero")
    .dropDuplicates()
)

janela_generos = Window.orderBy("nome_genero")

df_dim_genres = df_dim_genres.withColumn(
    "sk_genre_id",
    F.monotonically_increasing_id()
)

df_dim_genres = df_dim_genres.select(
    "sk_genre_id",
    "nome_genero"
)

display(df_dim_genres)
print(df_dim_genres.dtypes)

df_dim_genres.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_genres")

## 3. Criação da dimensão de Pessoas
---

In [0]:
# Selecionar apenas pessoas físicas envolvidas nos filmes.
df_dim_people = (
    spark.table("workspace.silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(
        F.col("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    )
    .dropDuplicates()
)

# Criar a chave substituta da dimensão.
df_dim_people = df_dim_people.withColumn(
    "sk_person_id",
    F.monotonically_increasing_id()
)

df_dim_people = df_dim_people.select(
    "sk_person_id",
    "nome_pessoa",
    "tipo_pessoa"
)

display(df_dim_people)
print(df_dim_people.dtypes)

df_dim_people.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_people")

## 4. Criação da dimensão de Produtores
---

In [0]:
# Selecionar apenas as entidades classificadas como Produtora.
df_dim_companies = (
    spark.table("workspace.silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(
        F.col("nome_entidade").alias("nome_produtora")
    )
    .dropDuplicates()
)

# Criar a chave substituta da dimensão.
df_dim_companies = df_dim_companies.withColumn(
    "sk_company_id",
    F.monotonically_increasing_id()
)

df_dim_companies = df_dim_companies.select(
    "sk_company_id",
    "nome_produtora"
)

display(df_dim_companies)
print(df_dim_companies.dtypes)

df_dim_companies.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_companies")

## 5. Criação da dimensão de Avaliações
---

In [0]:
# Carregar a tabela da camada Silver
df_reviews = spark.table("workspace.silver.tb_avaliacoes_usuarios")

# Agrupar as avaliações por filme, calculando a quantidade de avaliações e a nota média.
df_reviews = (
    df_reviews
    .groupBy("id_filme")
    .agg(
        F.count("*").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).alias("nota_media_usuarios")
    )
)

# Recuperamos a chave substituta de cada filme a partir da dimensão de filmes.
df_movies_key = (
    spark.table("workspace.gold.dim_movies")
    .select(
        "sk_movie_id",
        "id_filme"
    )
)

# Agora associamos cada avaliação resumida com a chave substituta do filme.
df_reviews = (
    df_reviews
    .join(
        df_movies_key,
        on="id_filme",
        how="inner"
    )
)

# Criação da chave substituta da dimensão de avaliações.
df_reviews = df_reviews.withColumn(
    "sk_review_id",
    F.monotonically_increasing_id()
)

df_reviews = df_reviews.select(
    "sk_review_id",
    "sk_movie_id",
    "qtd_avaliacoes_usuarios",
    "nota_media_usuarios"
)

display(df_reviews)
print(df_reviews.dtypes)

df_reviews.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_reviews")

## 6. Criação da bridge entre filmes e gêneros
---

In [0]:
# Carregar a tabela da camada Silver
df_bridge_genre = spark.table("workspace.silver.tb_generos")

# Recuperar as surrogate keys dos filmes
df_movies_key = (
    spark.table("workspace.gold.dim_movies")
    .select("sk_movie_id", "id_filme")
)

# Recuperar as surrogate keys dos gêneros
df_genres_key = (
    spark.table("workspace.gold.dim_genres")
    .select("sk_genre_id", "nome_genero")
)

# Associamos cada filme ao respectivo gênero por meio das chaves substitutas
df_bridge_genre = (
    df_bridge_genre
    .join(df_movies_key, on="id_filme", how="inner")
    .join(df_genres_key, on="nome_genero", how="inner")
    .select(
        "sk_movie_id",
        "sk_genre_id"
    )
    .dropDuplicates()
)

display(df_bridge_genre)

df_bridge_genre.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.bridge_movie_genre")

## 7. Criação da bridge entre filmes e pessoas
---

In [0]:
# Selecionar apenas atores, diretores e roteiristas da Silver
df_bridge_person = (
    spark.table("workspace.silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
)

# Recuperar as surrogate keys das pessoas
df_people_key = (
    spark.table("workspace.gold.dim_people")
    .select(
        "sk_person_id",
        F.col("nome_pessoa").alias("nome_entidade"),
        F.col("tipo_pessoa").alias("tipo_entidade")
    )
)

# Associamos filmes às respectivas pessoas
df_bridge_person = (
    df_bridge_person
    .join(df_movies_key, on="id_filme", how="inner")
    .join(
        df_people_key,
        on=["nome_entidade", "tipo_entidade"],
        how="inner"
    )
    .select(
        "sk_movie_id",
        "sk_person_id"
    )
    .dropDuplicates()
)

display(df_bridge_person)

df_bridge_person.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.bridge_movie_person")

## 8. Criação da bridge entre filmes e produtoras
---

In [0]:
# Selecionar apenas as produtoras da Silver
df_bridge_company = (
    spark.table("workspace.silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
)

# Recuperar as surrogate keys das produtoras
df_companies_key = (
    spark.table("workspace.gold.dim_companies")
    .select(
        "sk_company_id",
        F.col("nome_produtora").alias("nome_entidade")
    )
)

# Associamos filmes às respectivas produtoras
df_bridge_company = (
    df_bridge_company
    .join(df_movies_key, on="id_filme", how="inner")
    .join(df_companies_key, on="nome_entidade", how="inner")
    .select(
        "sk_movie_id",
        "sk_company_id"
    )
    .dropDuplicates()
)

display(df_bridge_company)

df_bridge_company.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.bridge_movie_company")

## 9. Criação da tabela fato de performance dos filmes
---

In [0]:
# Carregar a dimensão de filmes para obter a surrogate key
df_fact = (
    spark.table("workspace.gold.dim_movies")
    .select(
        "sk_movie_id",
        "id_filme",
        "status_filme"
    )
)

# Mantemos apenas filmes lançados
df_fact = df_fact.filter(
    F.col("status_filme") == "Lançado"
)

# Carregar as métricas financeiras da Silver
df_financeiro = (
    spark.table("workspace.silver.tb_financeiro_filmes")
    .select(
        "id_filme",
        "orcamento_usd",
        "receita_usd",
        "lucro_usd",
        "orcamento_brl",
        "receita_brl",
        "lucro_brl"
    )
)

# Carregar as métricas de engajamento da Silver
df_engajamento = (
    spark.table("workspace.silver.tb_metricas_engajamento")
    .select(
        "id_filme",
        "popularidade",
        "nota_media_tmdb",
        "qtd_votos_tmdb",
        "nota_media_imdb",
        "qtd_votos_imdb"
    )
)

# Unir as métricas financeiras e de engajamento à dimensão de filmes
df_fact = (
    df_fact
    .join(
        df_financeiro,
        on="id_filme",
        how="left"
    )
    .join(
        df_engajamento,
        on="id_filme",
        how="left"
    )
)

# Selecionamos as colunas finais da tabela fato.
# O id_filme é utilizado apenas durante os joins e não faz parte da fato final.

df_fact = df_fact.select(
    "sk_movie_id",
    "orcamento_usd",
    "receita_usd",
    "lucro_usd",
    "orcamento_brl",
    "receita_brl",
    "lucro_brl",
    "popularidade",
    "nota_media_tmdb",
    "qtd_votos_tmdb",
    "nota_media_imdb",
    "qtd_votos_imdb"
)

display(df_fact)

df_fact.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.fact_movies_performance")
